In [105]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from delta import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window

warehouse_location = 'hdfs://hdfs-nn:9000/warehouse'

builder = SparkSession \
    .builder \
    .appName("Fase Gold - Adaptacao Literaria") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport() \

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [106]:
spark.sql("DROP TABLE IF EXISTS gold_projeto.fact_adaptacao_literaria")

DataFrame[]

In [107]:
# Schema da tabela
spark.sql("""
    CREATE EXTERNAL TABLE IF NOT EXISTS gold_projeto.fact_adaptacao_literaria (
    id_adaptation BIGINT,
    fk_content INT,
    fk_release_date INT,
    book_title STRING,
    book_author STRING,
    book_publish_year INT,
    content_release_year INT,
    years_until_adaptation INT,
    runtime_minutes INT,
    number_of_seasons DOUBLE,
    content_type STRING
    )
    USING DELTA
    LOCATION 'hdfs://hdfs-nn:9000/gold/gold_projeto.db/fact_adaptacao_literaria'
""")

DataFrame[]

In [108]:
# Carregar Tabelas Necessárias
df_dim_conteudo = spark.table("gold_projeto.dim_conteudo")
df_dim_tempo = spark.table("gold_projeto.dim_tempo")
df_book_to_imdb = spark.table("projeto.book_to_imdb")
df_book_movie_adapt = spark.table("projeto.book_movie_adapt")

In [109]:
# Preparar e Cruzar Dados

# Identificar Conteúdos que são Adaptações (Via book_to_imdb)
df_book_to_imdb_clean = df_book_to_imdb.dropDuplicates(["imdbId", "bookLabel", "authorLabel"])

df_adaptacoes_base = df_dim_conteudo.join(
    df_book_to_imdb_clean,
    df_dim_conteudo.imdb_id == df_book_to_imdb_clean.imdbId,
    "inner"
).select(
    df_dim_conteudo["id_conteudo"],
    df_dim_conteudo["title"],
    df_dim_conteudo["release_year"],
    df_dim_conteudo["type"],
    df_dim_conteudo["runtime"],
    df_dim_conteudo["seasons"],
    df_book_to_imdb_clean["bookLabel"],
    df_book_to_imdb_clean["authorLabel"]
)

# Enriquecer com Ano do Livro (Via book_movie_adapt)
df_book_metadata_unique = df_book_movie_adapt.select(
        "bookLabel", 
        "authorLabel", 
        "book_year"
    ).dropDuplicates(["bookLabel", "authorLabel"]) 

# Join com normalização de texto para garantir match
df_adapt_enriquecida = df_adaptacoes_base.join(
    df_book_metadata_unique,
    (F.lower(F.trim(df_adaptacoes_base.bookLabel)) == F.lower(F.trim(df_book_metadata_unique.bookLabel))) &
    (F.lower(F.trim(df_adaptacoes_base.authorLabel)) == F.lower(F.trim(df_book_metadata_unique.authorLabel))),
    "left"
).select(
    df_adaptacoes_base["*"],
    df_book_metadata_unique["book_year"].cast("int").alias("ano_livro")
)

In [110]:
# Join com Dim_Tempo e Seleção Final

# Criamos uma data de referência (01-01-ANO) para ligar à dim_tempo
df_prep_tempo = df_adapt_enriquecida.withColumn(
    "data_referencia", 
    F.to_date(F.concat(F.col("release_year"), F.lit("-01-01")))
)

df_fact_calc = df_prep_tempo.join(
    df_dim_tempo,
    df_prep_tempo.data_referencia == df_dim_tempo.data,
    "left"
).select(
    # --- FKs ---
    F.col("id_conteudo").alias("fk_content"),
    F.col("id_data").alias("fk_release_date"),
    
    # --- Dimensões Degeneradas ---
    F.col("bookLabel").alias("book_title"),
    F.col("authorLabel").alias("book_author"),
    F.col("ano_livro").alias("book_publish_year"),
    F.col("release_year").alias("content_release_year"),
    F.col("type").alias("content_type"),
    
    # --- Metricas ---
    F.when(
        (F.col("ano_livro").isNotNull()) & (F.col("release_year") >= F.col("ano_livro")),
        F.col("release_year") - F.col("ano_livro")
    ).otherwise(None).alias("years_until_adaptation"),

    F.col("runtime").alias("runtime_minutes"),
    F.col("seasons").alias("number_of_seasons")
)

# Limpeza Final
df_fact_final = df_fact_calc.dropDuplicates()

In [111]:
# Gerar chave primária sequencial
windowSpec = Window.orderBy("fk_release_date", "fk_content")

df_final_pk = df_fact_final.withColumn(
    "id_adaptation", 
    F.row_number().over(windowSpec)
)

In [112]:
df_final_pk.printSchema()
df_final_pk.toPandas()

root
 |-- fk_content: integer (nullable = true)
 |-- fk_release_date: integer (nullable = true)
 |-- book_title: string (nullable = true)
 |-- book_author: string (nullable = true)
 |-- book_publish_year: integer (nullable = true)
 |-- content_release_year: integer (nullable = true)
 |-- content_type: string (nullable = true)
 |-- years_until_adaptation: integer (nullable = true)
 |-- runtime_minutes: integer (nullable = true)
 |-- number_of_seasons: integer (nullable = true)
 |-- id_adaptation: integer (nullable = false)



,fk_content,fk_release_date,book_title,book_author,book_publish_year,content_release_year,content_type,years_until_adaptation,runtime_minutes,number_of_seasons,id_adaptation
0,10881,355,the greatest gift,philip van doren stern,1945.0,1946,movie,1.0,130,0,1
1,1799,675,paths of glory,humphrey cobb,1935.0,1957,movie,22.0,88,0,2
2,12830,1232,lupin iii,monkey punch,NaN,1971,show,NaN,23,6,3
3,2746,1279,deliverance,james dickey,1970.0,1972,movie,2.0,109,0,4
4,1906,1426,arthurian romance,NA,NaN,1975,movie,NaN,91,0,5
...,...,...,...,...,...,...,...,...,...,...,...
255,14041,7081,the office blind date,NA,NaN,2022,show,NaN,61,1,256
256,14090,7081,the andy warhol diaries,andy warhol,NaN,2022,show,NaN,66,1,257
257,14136,7081,kotaro lives alone,mami tsumura,NaN,2022,show,NaN,27,1,258
258,14185,7081,q97628048,NA,NaN,2022,show,NaN,41,1,259


In [116]:
df_final_pk.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("hdfs://hdfs-nn:9000/gold/gold_projeto.db/fact_adaptacao_literaria")

In [117]:
# Validação e Análise Exploratória
print("\n--- Estatísticas das Métricas ---")
df_final_pk.select(
    F.avg("years_until_adaptation").alias("Media_Anos_Espera"),
    F.max("years_until_adaptation").alias("Max_Anos_Espera"),
    F.avg("runtime_minutes").alias("Media_Duracao"),
    F.avg("number_of_seasons").alias("Media_Temporadas")
).show()

print("\n--- Adaptações mais Rápidas (Top 5 Autores) ---")
df_final_pk.filter(
    (F.col("years_until_adaptation").isNotNull()) & 
    (F.col("book_author") != "NA") & 
    (F.col("book_author").isNotNull())
) \
    .groupBy("book_author") \
    .agg(F.avg("years_until_adaptation").alias("avg_years")) \
    .orderBy("avg_years") \
    .show(5, truncate=False)

print("\n--- Top 5 Autores com Mais Adaptações ---")
df_final_pk.filter(
    (F.col("book_author") != "NA") & 
    (F.col("book_author").isNotNull())
) \
    .groupBy("book_author") \
    .count() \
    .withColumnRenamed("count", "total_adaptacoes") \
    .orderBy(F.desc("total_adaptacoes")) \
    .show(5, truncate=False)


--- Estatísticas das Métricas ---
+-----------------+---------------+-----------------+------------------+
|Media_Anos_Espera|Max_Anos_Espera|    Media_Duracao|  Media_Temporadas|
+-----------------+---------------+-----------------+------------------+
|             36.5|            425|46.62692307692308|3.1423076923076922|
+-----------------+---------------+-----------------+------------------+


--- Adaptações mais Rápidas (Top 5 Autores) ---
+----------------------+---------+
|book_author           |avg_years|
+----------------------+---------+
|rex pickett           |0.0      |
|philip van doren stern|1.0      |
|naoki matayoshi       |1.0      |
|stephen king          |2.0      |
|james dickey          |2.0      |
+----------------------+---------+
only showing top 5 rows


--- Top 5 Autores com Mais Adaptações ---
+----------------+----------------+
|book_author     |total_adaptacoes|
+----------------+----------------+
|philippa gregory|5               |
|michael connelly|3    

In [119]:
# Verificar Resultados Finais
spark.table("gold_projeto.fact_adaptacao_literaria").show(10, truncate=False)
print(f"Total de registos: {spark.table('gold_projeto.fact_adaptacao_literaria').count()}")

+----------+---------------+-----------------------------+----------------------+-----------------+--------------------+------------+----------------------+---------------+-----------------+-------------+
|fk_content|fk_release_date|book_title                   |book_author           |book_publish_year|content_release_year|content_type|years_until_adaptation|runtime_minutes|number_of_seasons|id_adaptation|
+----------+---------------+-----------------------------+----------------------+-----------------+--------------------+------------+----------------------+---------------+-----------------+-------------+
|10881     |355            |the greatest gift            |philip van doren stern|1945             |1946                |movie       |1                     |130            |0                |1            |
|1799      |675            |paths of glory               |humphrey cobb         |1935             |1957                |movie       |22                    |88             |0       

In [10]:
spark.stop()